In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader


# ============================================================
# CNN MODEL
# ============================================================

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=32,
            kernel_size=3
        )

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3
        )

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        self.fc1 = nn.Linear(
            64 * 5 * 5,
            128
        )

        self.fc2 = nn.Linear(
            128,
            10
        )

    def forward(self, x):

        x = self.pool(
            F.relu(
                self.conv1(x)
            )
        )

        x = self.pool(
            F.relu(
                self.conv2(x)
            )
        )

        x = torch.flatten(
            x,
            1
        )

        x = F.relu(
            self.fc1(x)
        )

        return self.fc2(x)


# ============================================================
# DNN MODEL
# ============================================================

class SimpleDNN(nn.Module):
    def __init__(self):
        super(SimpleDNN, self).__init__()

        self.fc1 = nn.Linear(
            28 * 28,
            512
        )

        self.fc2 = nn.Linear(
            512,
            256
        )

        self.fc3 = nn.Linear(
            256,
            128
        )

        self.fc4 = nn.Linear(
            128,
            10
        )

    def forward(self, x):

        x = torch.flatten(
            x,
            1
        )

        x = F.relu(
            self.fc1(x)
        )

        x = F.relu(
            self.fc2(x)
        )

        x = F.relu(
            self.fc3(x)
        )

        return self.fc4(x)


# ============================================================
# SHARED PREPROCESSING
# ============================================================
# MNIST, FashionMNIST, and KMNIST are all 28x28 grayscale with
# 10 classes, so the same transform and model architectures work
# for all three. Each dataset gets its own normalization stats.

TRANSFORMS = {
    "MNIST": transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ]),
    "FashionMNIST": transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.2860,), (0.3530,)),
    ]),
    "KMNIST": transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1918,), (0.3483,)),
    ]),
}


# ============================================================
# DATASET CLASSES (torchvision)
# ============================================================

DATASET_CLASSES = {
    "MNIST": datasets.MNIST,
    "FashionMNIST": datasets.FashionMNIST,
    "KMNIST": datasets.KMNIST,  # Kuzushiji-MNIST
}


# ============================================================
# MODEL CLASSES
# ============================================================

MODEL_CLASSES = {
    "CNN": SimpleCNN,
    "DNN": SimpleDNN,
}


# ============================================================
# DATA LOADER BUILDER
# ============================================================

def build_train_loader(dataset_name, batch_size=64):

    dataset_cls = DATASET_CLASSES[dataset_name]
    transform = TRANSFORMS[dataset_name]

    train_dataset = dataset_cls(
        root="./classical_data",
        train=True,
        download=True,
        transform=transform
    )

    return DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True
    )


# ============================================================
# TRAINING ENGINE
# ============================================================

def run_experiment(
    model_class,
    model_name,
    dataset_name,
    train_loader,
    num_epochs=10
):

    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    print(f"\nUsing device: {device}")


    # --------------------------------------------------------
    # Create model
    # --------------------------------------------------------

    model = model_class().to(device)


    # --------------------------------------------------------
    # Optimizer
    # --------------------------------------------------------

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )


    # --------------------------------------------------------
    # Loss
    # --------------------------------------------------------

    criterion = nn.CrossEntropyLoss()


    # ========================================================
    # Training
    # ========================================================

    print(f"\nStarting {model_name} training on {dataset_name}...")

    model.train()

    for epoch in range(num_epochs):

        running_loss = 0.0
        correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            # --------------------------------------------
            # Clear gradients
            # --------------------------------------------

            optimizer.zero_grad()

            # --------------------------------------------
            # Forward
            # --------------------------------------------

            outputs = model(images)

            # --------------------------------------------
            # Loss
            # --------------------------------------------

            loss = criterion(outputs, labels)

            # --------------------------------------------
            # Backward
            # --------------------------------------------

            loss.backward()

            # --------------------------------------------
            # Update weights
            # --------------------------------------------

            optimizer.step()

            # --------------------------------------------
            # Statistics
            # --------------------------------------------

            running_loss += loss.item()

            predicted = torch.argmax(outputs, dim=1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        # ====================================================
        # Epoch metrics
        # ====================================================

        epoch_accuracy = 100.0 * correct / total
        average_loss = running_loss / len(train_loader)

        print(
            f"[{dataset_name} | {model_name}] "
            f"Epoch [{epoch + 1}/{num_epochs}] "
            f"Loss: {average_loss:.4f} "
            f"Accuracy: {epoch_accuracy:.2f}%"
        )

    # ========================================================
    # Save model
    # ========================================================

    save_path = f"{dataset_name.lower()}_{model_name.lower()}.pth"

    torch.save(model.state_dict(), save_path)

    print(f"\n{dataset_name} - {model_name} training complete.")
    print(f"Weights saved to: {save_path}")


# ============================================================
# MAIN: RUN CNN + DNN ON MNIST, FASHIONMNIST, KMNIST
# ============================================================

if __name__ == "__main__":

    NUM_EPOCHS = 10
    BATCH_SIZE = 64

    for dataset_name in ["MNIST", "FashionMNIST", "KMNIST"]:

        train_loader = build_train_loader(
            dataset_name,
            batch_size=BATCH_SIZE
        )

        for model_name, model_class in MODEL_CLASSES.items():

            run_experiment(
                model_class=model_class,
                model_name=model_name,
                dataset_name=dataset_name,
                train_loader=train_loader,
                num_epochs=NUM_EPOCHS
            )

# Test

In [ ]:
import hashlib
import os
import platform
import socket
import sys
import time
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from tqdm import tqdm
import pandas as pd
import psutil
import torch
import torch.nn.functional as F
from torchvision import datasets, transforms
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)



class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.conv1 = nn.Conv2d(
            in_channels=1,
            out_channels=32,
            kernel_size=3
        )

        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3
        )

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        self.fc1 = nn.Linear(
            64 * 5 * 5,
            128
        )

        self.fc2 = nn.Linear(
            128,
            10
        )

    def forward(self, x):

        x = self.pool(
            F.relu(
                self.conv1(x)
            )
        )

        x = self.pool(
            F.relu(
                self.conv2(x)
            )
        )

        x = torch.flatten(
            x,
            1
        )

        x = F.relu(
            self.fc1(x)
        )

        return self.fc2(x)


# ============================================================
# DNN MODEL
# ============================================================

class SimpleDNN(nn.Module):
    def __init__(self):
        super(SimpleDNN, self).__init__()

        self.fc1 = nn.Linear(
            28 * 28,
            512
        )

        self.fc2 = nn.Linear(
            512,
            256
        )

        self.fc3 = nn.Linear(
            256,
            128
        )

        self.fc4 = nn.Linear(
            128,
            10
        )

    def forward(self, x):

        x = torch.flatten(
            x,
            1
        )

        x = F.relu(
            self.fc1(x)
        )

        x = F.relu(
            self.fc2(x)
        )

        x = F.relu(
            self.fc3(x)
        )

        return self.fc4(x)

#from utils.llm_utils import load_tiny_llm_model, predict_with_tiny_llm
#from utils.vlm_utils import load_tiny_vlm_model, predict_with_tiny_vlm



def get_cpu_model():
    try:
        import cpuinfo
        return cpuinfo.get_cpu_info().get("brand_raw", "Unknown")
    except Exception:
        return platform.processor() or "Unknown"

CPU_MODEL_NAME = get_cpu_model()
def make_stable_device_id():
    raw = f"{socket.gethostname()}-{platform.system()}-{platform.machine()}-{CPU_MODEL_NAME}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()

DEVICE_UUID = make_stable_device_id()
DEVICE_SHORT = DEVICE_UUID[:8]

OUTPUT_ROOT = Path.cwd() / "test_results"
OUTPUT_ROOT.mkdir(exist_ok=True)
DEVICE_LOG_DIR = OUTPUT_ROOT / f"{DEVICE_SHORT}"
DEVICE_LOG_DIR.mkdir(exist_ok=True)


# =============================================================================
# Retrieves env var 'NUM_TEST_SAMPLES', defaults to 10 if not set
NUM_TEST_SAMPLES = int(os.getenv("NUM_TEST_SAMPLES", 10))

def get_cpu_model():
    try:
        import cpuinfo
        return cpuinfo.get_cpu_info().get("brand_raw", "Unknown")
    except Exception:
        return platform.processor() or "Unknown"

CPU_MODEL_NAME = get_cpu_model()
def make_stable_device_id():
    raw = f"{socket.gethostname()}-{platform.system()}-{platform.machine()}-{CPU_MODEL_NAME}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


DEVICE_UUID = make_stable_device_id()
DEVICE_SHORT = DEVICE_UUID[:8]

OUTPUT_ROOT = Path.cwd() / "test_results"
OUTPUT_ROOT.mkdir(exist_ok=True)
DEVICE_LOG_DIR = OUTPUT_ROOT / f"{DEVICE_SHORT}"
DEVICE_LOG_DIR.mkdir(exist_ok=True)

DATA_ROOT = Path("./classical_data")

# VERBOSE_DATASET_PATH = Path(
#     "mnist_fgsm_test_dataset.pt"
# )

# ============================================================
# Dataset configuration: MNIST, FashionMNIST, Kuzushiji-MNIST
# ============================================================
# Each dataset uses its own torchvision class, its own per-channel
# normalization stats, and its own saved model checkpoint prefix
# (produced by the training script as "<name>_cnn.pth" / "<name>_dnn.pth").

DATASET_CONFIGS = {
    "MNIST": {
        "dataset_class": datasets.MNIST,
        "mean": (0.1307,),
        "std": (0.3081,),
        "checkpoint_prefix": "mnist",
    },
    "FashionMNIST": {
        "dataset_class": datasets.FashionMNIST,
        "mean": (0.2860,),
        "std": (0.3530,),
        "checkpoint_prefix": "fashionmnist",
    },
    "KMNIST": {
        "dataset_class": datasets.KMNIST,  # Kuzushiji-MNIST
        "mean": (0.1918,),
        "std": (0.3483,),
        "checkpoint_prefix": "kmnist",
    },
}

# =============================================================================



torch.set_grad_enabled(False)



DEVICE_MODE = os.getenv("DEVICE_MODE", "cpu")  # "cpu" | "cuda" | "auto"


def resolve_device():
    if DEVICE_MODE.lower() == "cpu":
        return torch.device("cpu")
    if DEVICE_MODE.lower() == "cuda":
        if torch.cuda.is_available():
            return torch.device("cuda")
        print("CUDA requested but not available. Falling back to CPU.")
        return torch.device("cpu")
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = torch.device("cpu")#resolve_device()


# ============================================================
# Safe optional imports
# ============================================================

# ---- CodeCarbon ----
try:
    from codecarbon import EmissionsTracker
    import codecarbon
    CODECARBON_AVAILABLE = True
    CODECARBON_VERSION = codecarbon.__version__
except Exception:
    EmissionsTracker = None
    CODECARBON_AVAILABLE = False
    CODECARBON_VERSION = "unavailable"
    print("CodeCarbon not available. Energy values will be set to 0.")

# ---- pynvml ----
try:
    import pynvml
    pynvml.nvmlInit()
    NVML_AVAILABLE = True
    NVML_HANDLE = pynvml.nvmlDeviceGetHandleByIndex(0) if torch.cuda.is_available() else None
except Exception:
    NVML_AVAILABLE = False
    NVML_HANDLE = None
    print("pynvml not available.")

# ---- py-cpuinfo ----
try:
    import cpuinfo
    _CPU_INFO = cpuinfo.get_cpu_info()
    CPU_MODEL_NAME = _CPU_INFO.get("brand_raw", "Unknown")
    CPU_ARCH = _CPU_INFO.get("arch", platform.machine())
    CPU_TDP_W = None
except Exception:
    CPU_MODEL_NAME = "Unknown"
    CPU_ARCH = platform.machine()
    CPU_TDP_W = None
    print("cpuinfo not available.")

# ---- fvcore: FLOPs ----
try:
    from fvcore.nn import FlopCountAnalysis
    FVCORE_AVAILABLE = True
except Exception:
    FlopCountAnalysis = None
    FVCORE_AVAILABLE = False
    print("fvcore not available.")


# ============================================================
# OS / environment constants
# ============================================================

def get_os_full_name():
    system = platform.system()
    architecture = platform.machine()

    if system == "Windows":
        try:
            import winreg
            key = winreg.OpenKey(
                winreg.HKEY_LOCAL_MACHINE,
                r"SOFTWARE\Microsoft\Windows NT\CurrentVersion"
            )
            product_name = winreg.QueryValueEx(key, "ProductName")[0]
            display_version = winreg.QueryValueEx(key, "DisplayVersion")[0]
            current_build = winreg.QueryValueEx(key, "CurrentBuild")[0]
            return f"{product_name} {display_version} Build {current_build} {architecture}"
        except Exception:
            return f"Windows {platform.release()} {architecture}"

    if system == "Linux":
        os_info = {}
        try:
            with open("/etc/os-release", "r", encoding="utf-8") as f:
                for line in f:
                    if "=" in line:
                        k, v = line.strip().split("=", 1)
                        os_info[k] = v.strip('"')
        except Exception:
            pass
        pretty_name = os_info.get("PRETTY_NAME")
        name = os_info.get("NAME")
        version = os_info.get("VERSION")
        version_id = os_info.get("VERSION_ID")
        distro_id = os_info.get("ID")
        if pretty_name:
            return f"{pretty_name} {architecture}"
        if name and version:
            return f"{name} {version} {architecture}"
        if name and version_id:
            return f"{name} {version_id} {architecture}"
        if distro_id:
            return f"{distro_id} {platform.release()} {architecture}"
        return f"Linux {platform.release()} {architecture}"

    if system == "Darwin":
        return f"macOS {platform.mac_ver()[0]} {architecture}"

    return f"{system} {platform.release()} {architecture}"


TORCH_VERSION = torch.__version__
PYTHON_VERSION = sys.version.split()[0]
OS_NAME = platform.system()
OS_VERSION = platform.version()
OS_ARCHITECTURE = platform.machine()
OS_FULL_NAME = get_os_full_name()
SYSTEM_RAM_TOTAL_GB = round(psutil.virtual_memory().total / (1024 ** 3), 2)
CPU_CORE_COUNT = psutil.cpu_count(logical=False)
CPU_THREAD_COUNT = psutil.cpu_count(logical=True)


# ============================================================
# Stable device ID (SHA256-based, consistent across runs)
# ============================================================

def make_stable_device_id():
    raw = f"{socket.gethostname()}-{platform.system()}-{platform.machine()}-{CPU_MODEL_NAME}"
    print(f"Generating device ID from: {raw}")
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


DEVICE_UUID = make_stable_device_id()
DEVICE_SHORT = DEVICE_UUID[:8]


# ============================================================
# GPU static info
# ============================================================

def _get_cuda_driver_version():
    if not NVML_AVAILABLE:
        return None
    try:
        driver = pynvml.nvmlSystemGetDriverVersion()
        return driver.decode("utf-8") if isinstance(driver, bytes) else driver
    except Exception:
        return None


CUDA_DRIVER_VERSION = _get_cuda_driver_version()


def _get_gpu_static():
    defaults = {
        "gpu_power_limit_w": None,
        "gpu_driver_version": CUDA_DRIVER_VERSION,
        "gpu_memory_total_mb": None,
        "gpu_compute_capability": None,
    }
    if not NVML_AVAILABLE or NVML_HANDLE is None:
        return defaults
    try:
        power_limit_mw = pynvml.nvmlDeviceGetPowerManagementLimit(NVML_HANDLE)
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
        cc_major, cc_minor = pynvml.nvmlDeviceGetCudaComputeCapability(NVML_HANDLE)
        return {
            "gpu_power_limit_w": round(power_limit_mw / 1000.0, 1),
            "gpu_driver_version": CUDA_DRIVER_VERSION,
            "gpu_memory_total_mb": round(mem_info.total / (1024 ** 2), 2),
            "gpu_compute_capability": f"{cc_major}.{cc_minor}",
        }
    except Exception:
        return defaults


GPU_STATIC = _get_gpu_static()


def _get_gpu_core_thread():
    """
    Return (gpu_core_count, gpu_thread_count) where:
      - gpu_core_count  = total CUDA cores  (multiprocessor_count * cores_per_sm)
      - gpu_thread_count = max threads per device (gpu_core_count * max_threads_per_block,
                           capped to a sensible ceiling via device properties)
    Falls back to torch.cuda device properties when pynvml SM count is unavailable.
    """
    if not torch.cuda.is_available():
        return None, None

    try:
        props = torch.cuda.get_device_properties(0)
        sm_count = props.multi_processor_count

        # Cores-per-SM lookup by compute capability major version
        cc_major = props.major
        cores_per_sm_map = {
            2: 32,   # Fermi
            3: 192,  # Kepler
            5: 128,  # Maxwell
            6: 64,   # Pascal (GP100=64, GP10x=128 — use 64 as conservative default)
            7: 64,   # Volta / Turing
            8: 128,  # Ampere
            9: 128,  # Ada Lovelace / Hopper
        }
        cores_per_sm = cores_per_sm_map.get(cc_major, 64)
        gpu_core_count = sm_count * cores_per_sm

        # gpu_thread_count = cores * max_threads_per_multiprocessor
        gpu_thread_count = sm_count * props.max_threads_per_multi_processor

        return gpu_core_count, gpu_thread_count

    except Exception:
        return None, None


GPU_CORE_COUNT, GPU_THREAD_COUNT = _get_gpu_core_thread()


# ============================================================
# Per-sample hardware helpers
# ============================================================

def get_hostname():
    return socket.gethostname()


def get_gpu_name():
    try:
        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
        return "No GPU"
    except Exception:
        return "Unknown"


def get_cpu_usage():
    return psutil.cpu_percent(interval=None)


def get_ram_usage():
    return psutil.virtual_memory().percent


def get_cpu_freq():
    try:
        freq = psutil.cpu_freq()
        return round(freq.current, 2) if freq else None
    except Exception:
        return None


def get_memory_footprint_mb():
    try:
        return round(psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024), 4)
    except Exception:
        return None


def get_gpu_metrics():
    null = {
        "gpu_power_draw_w": None,
        "gpu_utilization_pct": None,
        "gpu_temp_c": None,
        "gpu_memory_used_mb": None,
        "gpu_sm_clock_mhz": None,
        "gpu_memory_clock_mhz": None,
    }
    if not NVML_AVAILABLE or NVML_HANDLE is None:
        return null
    try:
        power_mw = pynvml.nvmlDeviceGetPowerUsage(NVML_HANDLE)
        util = pynvml.nvmlDeviceGetUtilizationRates(NVML_HANDLE)
        temp = pynvml.nvmlDeviceGetTemperature(NVML_HANDLE, pynvml.NVML_TEMPERATURE_GPU)
        mem_info = pynvml.nvmlDeviceGetMemoryInfo(NVML_HANDLE)
        sm_clock = pynvml.nvmlDeviceGetClockInfo(NVML_HANDLE, pynvml.NVML_CLOCK_SM)
        mem_clock = pynvml.nvmlDeviceGetClockInfo(NVML_HANDLE, pynvml.NVML_CLOCK_MEM)
        return {
            "gpu_power_draw_w": round(power_mw / 1000.0, 2),
            "gpu_utilization_pct": util.gpu,
            "gpu_temp_c": temp,
            "gpu_memory_used_mb": round(mem_info.used / (1024 ** 2), 2),
            "gpu_sm_clock_mhz": sm_clock,
            "gpu_memory_clock_mhz": mem_clock,
        }
    except Exception:
        return null


def get_cpu_temp():
    try:
        temps = psutil.sensors_temperatures()
        if not temps:
            return None
        for key in ("coretemp", "k10temp", "cpu_thermal", "acpitz"):
            if key in temps:
                values = [e.current for e in temps[key] if e.current and e.current > 0]
                if values:
                    return round(sum(values) / len(values), 1)
    except Exception:
        pass
    return None


def get_cpu_power_draw_w():
    """Stub — populate with platform-specific implementation if available."""
    return None


def get_cpu_cores_used():
    try:
        return sum(1 for p in psutil.cpu_percent(percpu=True) if p > 1.0)
    except Exception:
        return None


# ============================================================
# Prediction quality helpers
# ============================================================

def get_prediction_quality(logits):
    """Return (confidence, logit_margin, entropy) from a raw logits tensor."""
    probs = F.softmax(logits, dim=-1).squeeze()
    confidence = float(probs.max().item())
    top2 = torch.topk(logits.squeeze(), k=2).values
    margin = float((top2[0] - top2[1]).item())
    entropy = float(-(probs * torch.log(probs + 1e-12)).sum().item())
    return round(confidence, 6), round(margin, 6), round(entropy, 6)


# ============================================================
# FLOPs helper
# ============================================================

def compute_model_flops(model, device, input_shape=(1, 1, 28, 28)):
    if not FVCORE_AVAILABLE:
        return None
    try:
        dummy = torch.ones(input_shape, dtype=torch.float32, device=device)
        fc = FlopCountAnalysis(model, dummy)
        fc.unsupported_ops_warnings(False)
        fc.uncalled_modules_warnings(False)
        return int(fc.total())
    except Exception as e:
        print(f"[FLOPs unavailable] {e}")
        return None


# ============================================================
# Energy tracking
# ============================================================

def _extract_energy_data(tracker, emissions_value):
    fd = getattr(tracker, "final_emissions_data", None)
    cpu_energy = getattr(fd, "cpu_energy", 0) if fd else 0
    gpu_energy = getattr(fd, "gpu_energy", 0) if fd else 0
    ram_energy = getattr(fd, "ram_energy", 0) if fd else 0
    total_energy = getattr(fd, "energy_consumed", 0) if fd else 0
    carbon_intensity = None
    if emissions_value and total_energy and total_energy > 0:
        carbon_intensity = round(emissions_value / total_energy, 8)
    return cpu_energy, gpu_energy, ram_energy, total_energy, carbon_intensity


def run_with_energy_tracking(inference_fn, *args, output_dir="./test_results/codecarbon", **kwargs):
    os.makedirs(output_dir, exist_ok=True)

    if CODECARBON_AVAILABLE:
        tracker = EmissionsTracker(
            project_name="mnist_edge_inference",
            output_dir=output_dir,
            output_file="codecarbon_dnn_cnn_edge.csv",
            log_level="error",
            save_to_file=True,
        )
        tracker.start()
        t0 = time.perf_counter()
        result = inference_fn(*args, **kwargs)
        exec_time = time.perf_counter() - t0
        emissions_value = tracker.stop()
        gpu_snap = get_gpu_metrics()
        cpu_energy, gpu_energy, ram_energy, total_energy, carbon_intensity = \
            _extract_energy_data(tracker, emissions_value)
        return result, exec_time, cpu_energy, gpu_energy, ram_energy, total_energy, emissions_value, carbon_intensity, gpu_snap

    t0 = time.perf_counter()
    result = inference_fn(*args, **kwargs)
    exec_time = time.perf_counter() - t0
    gpu_snap = get_gpu_metrics()
    return result, exec_time, 0, 0, 0, 0, 0, None, gpu_snap


# ============================================================
# Dataset / CSV helpers
# ============================================================

def get_edge_dataset_path(file_name, dataset_name="MNIST"):
    # raw_dir = DEVICE_LOG_DIR 
    # raw_dir.mkdir(parents=True, exist_ok=True)
    return DEVICE_LOG_DIR / f"dnn_cnn_dataset_{DEVICE_SHORT}_{dataset_name.lower()}_{file_name}.csv"


def append_rows(rows, file_path):
    if not rows:
        return
    new_df = pd.DataFrame(rows)
    if file_path.exists():
        try:
            existing_df = pd.read_csv(file_path, on_bad_lines="skip")
            for col in new_df.columns:
                if col not in existing_df.columns:
                    existing_df[col] = None
            for col in existing_df.columns:
                if col not in new_df.columns:
                    new_df[col] = None
            new_df = new_df[existing_df.columns]
            pd.concat([existing_df, new_df], ignore_index=True).to_csv(file_path, index=False)
            return
        except Exception:
            pass
    new_df.to_csv(file_path, index=False)


def get_existing_count(file_path, model_name):
    if not file_path.exists():
        return 0
    try:
        df = pd.read_csv(file_path, on_bad_lines="skip")
        if "model_type" not in df.columns:
            return 0
        if "collection_mode" in df.columns:
            mask = (
                (df["model_type"].astype(str).str.strip() == model_name)
                & (df["collection_mode"].astype(str).str.strip() == "automated_edge")
            )
            return int(mask.sum())
        return int((df["model_type"].astype(str).str.strip() == model_name).sum())
    except Exception:
        return 0


# ============================================================
# Inference runners (unified)
# ============================================================

def _run_torch_model(model, image_tensor, mean=(0.1307,), std=(0.3081,)):
    """Shared inference path for CNN and DNN. `mean`/`std` are the
    dataset-specific normalization stats (MNIST, FashionMNIST, KMNIST each
    have their own)."""
    preprocess = transforms.Compose([
        transforms.Normalize(mean, std)
    ])
    input_tensor = preprocess(image_tensor).unsqueeze(0).to(DEVICE)
    with torch.inference_mode():
        logits = model(input_tensor)
        pred = torch.argmax(logits, 1).item()
    return pred, logits


def tensor_to_canvas_array(image_tensor):
    img = image_tensor.squeeze(0).cpu().numpy() * 255.0
    return img.clip(0, 255).astype("uint8")


# ============================================================
# Row builder
# ============================================================

def build_row(
    model_name,
    prediction,
    exec_time,
    parameters,
    true_label,
    sample_index,
    logits=None,
    cpu_energy=0,
    gpu_energy=0,
    ram_energy=0,
    total_energy=0,
    emissions_value=0,
    carbon_intensity=None,
    gpu_metrics=None,
    model_flops=None,
    model_under_attack=0,
    dataset_name="MNIST"
):
    gpu_metrics = gpu_metrics or {}

    # Prediction quality
    confidence_score, logit_margin, entropy = (None, None, None)
    if logits is not None:
        confidence_score, logit_margin, entropy = get_prediction_quality(logits)

    # Energy-derived efficiency metrics
    total_energy = total_energy or 0.0
    cpu_energy = cpu_energy or 0.0
    gpu_energy = gpu_energy or 0.0
    ram_energy = ram_energy or 0.0

    input_tokens = 784  # 28x28 pixel proxy
    output_tokens = 1
    total_tokens = input_tokens + output_tokens

    joules_per_token = 0.0
    energy_per_token_kwh = 0.0
    watts_estimated = 0.0
    gpu_energy_pct = 0.0
    cpu_energy_pct = 0.0

    if total_energy > 0 and total_tokens > 0:
        energy_per_token_kwh = round(total_energy / total_tokens, 12)
        joules_per_token = round((total_energy * 3_600_000) / total_tokens, 6)
        if exec_time > 0:
            watts_estimated = round((total_energy * 3_600_000) / exec_time, 4)
        gpu_energy_pct = round((gpu_energy / total_energy) * 100, 2)
        cpu_energy_pct = round((cpu_energy / total_energy) * 100, 2)

    correct = None
    if true_label is not None and prediction is not None:
        correct = int(prediction) == int(true_label)

    return {
        # --- Identity ---
        "timestamp":                    time.strftime("%Y-%m-%d %H:%M:%S"),
        "unique_device_id":             DEVICE_UUID,
        "device_short_id":              DEVICE_SHORT,
        "pc_name":                      get_hostname(),
        "collection_mode":              "automated_edge",

        # --- Sample ---
        "sample_index":                 sample_index,
        "true_label":                   true_label,
        "prediction":                   prediction,
        "correct":                      correct,

        # --- Model identity ---
        "dataset":                      dataset_name,
        "model_type":                   model_name,
        "parameters":                   parameters,
        "model_flops":                  model_flops,

        # --- Prediction quality ---
        "confidence_score":             confidence_score,
        "logit_margin":                 logit_margin,
        "entropy":                      entropy,

        # --- Timing ---
        "execution_time_sec":           round(exec_time, 10),

        # --- CodeCarbon energy ---
        "cpu_energy_kwh":               cpu_energy,
        "gpu_energy_kwh":               gpu_energy,
        "ram_energy_kwh":               ram_energy,
        "total_energy_kwh":             total_energy,
        "total_emissions_kg":           emissions_value,
        "carbon_intensity_kgco2_kwh":   carbon_intensity,
        "codecarbon_version":           CODECARBON_VERSION,

        # --- Efficiency derived ---
        "input_tokens":                 input_tokens,
        "output_tokens":                output_tokens,
        "total_tokens":                 total_tokens,
        "tokens_per_second":            round(total_tokens / exec_time, 4) if exec_time > 0 else None,
        "joules_per_token":             joules_per_token,
        "energy_per_token_kwh":         energy_per_token_kwh,
        "watts_estimated":              watts_estimated,
        "gpu_energy_pct_of_total":      gpu_energy_pct,
        "cpu_energy_pct_of_total":      cpu_energy_pct,

        # --- CPU hardware ---
        "cpu_model":                    CPU_MODEL_NAME,
        "cpu_architecture":             CPU_ARCH,
        "cpu_core_count":               CPU_CORE_COUNT,
        "cpu_thread_count":             CPU_THREAD_COUNT,
        "cpu_core":                     CPU_CORE_COUNT,
        "cpu_thread":                   CPU_THREAD_COUNT,
        "cpu_tdp_w":                    CPU_TDP_W,
        "cpu_usage_pct":                get_cpu_usage(),
        "cpu_clock_mhz":                get_cpu_freq(),
        "cpu_temp_c":                   get_cpu_temp(),
        "cpu_power_draw_w":             get_cpu_power_draw_w(),
        "cpu_cores_used":               get_cpu_cores_used(),

        # --- GPU hardware ---
        "gpu_model":                    get_gpu_name(),
        "gpu_core":                     GPU_CORE_COUNT,
        "gpu_thread":                   GPU_THREAD_COUNT,
        "gpu_driver_version":           GPU_STATIC["gpu_driver_version"],
        "gpu_compute_capability":       GPU_STATIC["gpu_compute_capability"],
        "gpu_power_limit_w":            GPU_STATIC["gpu_power_limit_w"],
        "gpu_memory_total_mb":          GPU_STATIC["gpu_memory_total_mb"],
        "gpu_power_draw_w":             gpu_metrics.get("gpu_power_draw_w"),
        "gpu_utilization_pct":          gpu_metrics.get("gpu_utilization_pct"),
        "gpu_temp_c":                   gpu_metrics.get("gpu_temp_c"),
        "gpu_memory_used_mb":           gpu_metrics.get("gpu_memory_used_mb"),
        "gpu_sm_clock_mhz":             gpu_metrics.get("gpu_sm_clock_mhz"),
        "gpu_memory_clock_mhz":         gpu_metrics.get("gpu_memory_clock_mhz"),
        "cuda_driver_version":          CUDA_DRIVER_VERSION,
        "cuda_available":               torch.cuda.is_available(),
        "device_type":                  str(DEVICE),

        # --- RAM / memory ---
        "ram_usage_pct":                psutil.virtual_memory().percent,
        "memory_footprint_mb":          get_memory_footprint_mb(),
        "system_ram_total_gb":          SYSTEM_RAM_TOTAL_GB,

        # --- Environment ---
        "os_name":                      OS_NAME,
        "os_version":                   OS_VERSION,
        "os_architecture":              OS_ARCHITECTURE,
        "os_full_name":                 OS_FULL_NAME,
        "python_version":               PYTHON_VERSION,
        "torch_version":                TORCH_VERSION,

        # --- Final model metrics (backfilled after run) ---
        "model_accuracy":               None,
        "model_precision_weighted":     None,
        "model_recall_weighted":        None,
        "model_f1_weighted":            None,

        # --- custom metrics ---
        "quantum_computing":              model_under_attack,
    }



def backfill_model_metrics(file_path, model_name):
    """
    Compute model metrics from the prediction rows already saved in the
    telemetry CSV, then write those metrics back into the same CSV.
    No separate model_metrics.json file is used.
    """
    if not file_path.exists():
        return None

    df = pd.read_csv(file_path, on_bad_lines="skip")

    if "model_type" not in df.columns:
        return None

    mask = (
        df["model_type"]
        .astype(str)
        .str.strip()
        == model_name
    )

    model_df = df.loc[mask].copy()

    if model_df.empty:
        return None

    # Remove incomplete rows, if any.
    model_df = model_df.dropna(
        subset=["true_label", "prediction"]
    )

    if model_df.empty:
        return None

    y_true = model_df["true_label"].astype(int).tolist()
    y_pred = model_df["prediction"].astype(int).tolist()

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    precision_weighted = precision_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    recall_weighted = recall_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    f1_weighted = f1_score(
        y_true,
        y_pred,
        average="weighted",
        zero_division=0,
    )

    # Backfill all rows belonging to this model in the SAME CSV.
    df.loc[
        mask,
        "model_accuracy",
    ] = float(accuracy)

    df.loc[
        mask,
        "model_precision_weighted",
    ] = float(precision_weighted)

    df.loc[
        mask,
        "model_recall_weighted",
    ] = float(recall_weighted)

    df.loc[
        mask,
        "model_f1_weighted",
    ] = float(f1_weighted)

    df.to_csv(
        file_path,
        index=False,
    )

    metrics = {
        "accuracy": float(accuracy),
        "precision_weighted": float(precision_weighted),
        "recall_weighted": float(recall_weighted),
        "f1_weighted": float(f1_weighted),
    }

    print(
        f"{model_name} metrics -> "
        f"Accuracy: {accuracy:.4f}, "
        f"Precision(weighted): {precision_weighted:.4f}, "
        f"Recall(weighted): {recall_weighted:.4f}, "
        f"F1(weighted): {f1_weighted:.4f}"
    )

    return metrics


# ============================================================
# Main collection loop
# ============================================================

def collect_for_model(base_dataset, model_name, dataset_name="MNIST", mean=(0.1307,), std=(0.3081,), num_samples=250, flush_every=25, file_name ='vanilla'):
    print(f"\nCollecting {num_samples} edge samples for {dataset_name} / {model_name} on {get_hostname()}")
    print(f"Device UUID : {DEVICE_UUID}")
    print(f"Device short: {DEVICE_SHORT}")

    output_path = get_edge_dataset_path(file_name, dataset_name=dataset_name)

    rows = []

    checkpoint_prefix = DATASET_CONFIGS[dataset_name]["checkpoint_prefix"]

    if model_name == "CNN":
        model = SimpleCNN().to(DEVICE)
        model.load_state_dict(torch.load(f"{checkpoint_prefix}_cnn.pth", map_location=DEVICE))
        model.eval()
        model_flops = compute_model_flops(model, DEVICE)

    elif model_name == "DNN":
        model = SimpleDNN().to(DEVICE)
        model.load_state_dict(torch.load(f"{checkpoint_prefix}_dnn.pth", map_location=DEVICE))
        model.eval()
        model_flops = compute_model_flops(model, DEVICE)

    else:
        raise ValueError(f"Unknown model: {model_name}")

    # Compute parameter count directly from the loaded model.
    # No separate model-metrics file is needed.
    parameters = sum(
        p.numel()
        for p in model.parameters()
    )

    limit = min(num_samples, len(base_dataset))
    existing_count = get_existing_count(output_path, model_name)

    if existing_count >= limit:
        print(f"{model_name}: already complete ({existing_count}/{limit})")
        backfill_model_metrics(
            output_path,
            model_name,
        )
        return

    print(f"{model_name}: resuming from {existing_count}/{limit}")

    progress_bar = tqdm(
        range(existing_count, limit),
        desc=model_name,
        unit="sample",
        dynamic_ncols=True,
    )

    for i in progress_bar:
        image_tensor, true_label = base_dataset[i]

        logits = None

        if model_name in ("CNN", "DNN"):
            (pred, logits), exec_time, cpu_energy, gpu_energy, ram_energy, \
                total_energy, emissions_value, carbon_intensity, gpu_snap = \
                run_with_energy_tracking(
                    _run_torch_model, model, image_tensor, mean=mean, std=std
                )

        row = build_row(
            model_name=model_name,
            prediction=pred,
            exec_time=exec_time,
            parameters=parameters,
            true_label=int(true_label),
            sample_index=i,
            logits=logits,
            cpu_energy=cpu_energy,
            gpu_energy=gpu_energy,
            ram_energy=ram_energy,
            total_energy=total_energy,
            emissions_value=emissions_value,
            carbon_intensity=carbon_intensity,
            gpu_metrics=gpu_snap,
            model_flops=model_flops,
            model_under_attack= file_name != 'vanilla',
            dataset_name=dataset_name
        )

        rows.append(row)

        progress_bar.set_postfix({
            "pred": pred,
            "true": int(true_label),
            "time_s": round(exec_time, 3),
            "done": i + 1,
        })

        if (i + 1) % flush_every == 0:
            append_rows(rows, output_path)
            rows = []

    if rows:
        append_rows(rows, output_path)

    # Compute Accuracy / Precision / Recall / F1 from the saved predictions
    # and write them back into the SAME telemetry CSV.
    backfill_model_metrics(
        output_path,
        model_name,
    )

    print(f"{model_name}: finished {limit}/{limit} -> {output_path}")


def main():

    # Loop over all three datasets (MNIST, FashionMNIST, KMNIST) and both
    # architectures (CNN, DNN). Each combination loads its own checkpoint
    # (e.g. "fashionmnist_cnn.pth") and writes to its own CSV, so results
    # never overwrite or mix across datasets.

    for dataset_name, config in DATASET_CONFIGS.items():

        print(f"\n{'=' * 60}")
        print(f"Dataset: {dataset_name}")
        print(f"{'=' * 60}")

        # Raw ToTensor only here — normalization is applied per-sample
        # inside _run_torch_model using this dataset's own mean/std.
        base_dataset = config["dataset_class"](
            root="./classical_data",
            train=False,
            download=True,
            transform=transforms.ToTensor()
        )

        for model_name in ("CNN", "DNN"):
            collect_for_model(
                base_dataset,
                model_name,
                dataset_name=dataset_name,
                mean=config["mean"],
                std=config["std"],
                num_samples=NUM_TEST_SAMPLES,
                flush_every=25,
                file_name='vanilla'
            )

    print("\nDone.")


if __name__ == "__main__":
    main()


# Testing

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import platform
import re
import socket
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import pennylane as qml
import psutil
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm.auto import tqdm

# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42
SOURCE_QUBITS = 10
N_QUBITS = 1
N_CLASSES = 10
FIDELITY = "f90"
SAMPLES_PER_DATASET = 10  # 10 total/dataset; with 10 classes => one per class
PENNYLANE_DEVICE = "default.qubit"

PROJECT_ROOT = Path.cwd()
OUTPUT_ROOT = PROJECT_ROOT / "quantum_test_results"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Folder structure exactly matching the user's project screenshot.
# Fashion fallbacks are included because some earlier scripts used
# mnisq_fashion_data / mnisq_fashion_results_q1.
DATASET_CONFIGS = {
    "MNIST": {
        "data_roots": [
            PROJECT_ROOT / "mnisq_mnist_data",
        ],
        "result_roots": [
            PROJECT_ROOT / "mnisq_mnist_results_q1",
        ],
        "archive_tokens": ["base_test_mnist_784", "mnist_784"],
        "dataset_tokens": ["mnist_784"],
        "class_names": [str(i) for i in range(10)],
    },
    "FashionMNIST": {
        "data_roots": [
            PROJECT_ROOT / "mnisq_fashionmnist_data",
            PROJECT_ROOT / "mnisq_fashion_data",
        ],
        "result_roots": [
            PROJECT_ROOT / "mnisq_fashionmnist_results_q1",
            PROJECT_ROOT / "mnisq_fashion_results_q1",
        ],
        "archive_tokens": ["base_test_fashion-mnist", "fashion-mnist"],
        "dataset_tokens": ["fashion-mnist", "fashionmnist"],
        "class_names": [
            "T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
            "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
        ],
    },
    "Kuzushiji-MNIST": {
        "data_roots": [
            PROJECT_ROOT / "mnisq_kuzushiji_data",
        ],
        "result_roots": [
            PROJECT_ROOT / "mnisq_kuzushiji_results_q1",
        ],
        "archive_tokens": ["base_test_kuzushiji-mnist", "kuzushiji-mnist"],
        "dataset_tokens": ["kuzushiji-mnist", "kuzushiji"],
        "class_names": [f"Class {i}" for i in range(10)],
    },
}

np.random.seed(SEED)

# ============================================================
# OPTIONAL ENERGY TRACKING
# ============================================================

try:
    from codecarbon import EmissionsTracker
    import codecarbon
    CODECARBON_AVAILABLE = True
    CODECARBON_VERSION = codecarbon.__version__
except Exception:
    EmissionsTracker = None
    CODECARBON_AVAILABLE = False
    CODECARBON_VERSION = "unavailable"
    print("CodeCarbon not available. Energy values will be set to 0.")

# ============================================================
# HARDWARE / ENVIRONMENT FINGERPRINT
# ============================================================

def get_cpu_model():
    try:
        import cpuinfo
        return cpuinfo.get_cpu_info().get("brand_raw", "Unknown")
    except Exception:
        return platform.processor() or "Unknown"


CPU_MODEL_NAME = get_cpu_model()
CPU_ARCH = platform.machine()
CPU_CORE_COUNT = psutil.cpu_count(logical=False)
CPU_THREAD_COUNT = psutil.cpu_count(logical=True)
CPU_TDP_W = None
SYSTEM_RAM_TOTAL_GB = round(psutil.virtual_memory().total / (1024 ** 3), 2)
OS_NAME = platform.system()
OS_VERSION = platform.version()
OS_ARCHITECTURE = platform.machine()
PYTHON_VERSION = sys.version.split()[0]
PENNYLANE_VERSION = getattr(qml, "__version__", "unknown")


def get_os_full_name():
    system = platform.system()
    architecture = platform.machine()
    if system == "Windows":
        try:
            import winreg
            key = winreg.OpenKey(
                winreg.HKEY_LOCAL_MACHINE,
                r"SOFTWARE\Microsoft\Windows NT\CurrentVersion",
            )
            product_name = winreg.QueryValueEx(key, "ProductName")[0]
            display_version = winreg.QueryValueEx(key, "DisplayVersion")[0]
            current_build = winreg.QueryValueEx(key, "CurrentBuild")[0]
            return f"{product_name} {display_version} Build {current_build} {architecture}"
        except Exception:
            return f"Windows {platform.release()} {architecture}"
    if system == "Linux":
        return f"Linux {platform.release()} {architecture}"
    if system == "Darwin":
        return f"macOS {platform.mac_ver()[0]} {architecture}"
    return f"{system} {platform.release()} {architecture}"


OS_FULL_NAME = get_os_full_name()


def make_stable_device_id():
    raw = f"{socket.gethostname()}-{platform.system()}-{platform.machine()}-{CPU_MODEL_NAME}"
    return hashlib.sha256(raw.encode("utf-8")).hexdigest()


DEVICE_UUID = make_stable_device_id()
DEVICE_SHORT = DEVICE_UUID[:8]
DEVICE_LOG_DIR = OUTPUT_ROOT / DEVICE_SHORT / "_quantum"
DEVICE_LOG_DIR.mkdir(parents=True, exist_ok=True)


def get_hostname():
    return socket.gethostname()


def get_cpu_usage():
    try:
        return psutil.cpu_percent(interval=None)
    except Exception:
        return None


def get_ram_usage():
    try:
        return psutil.virtual_memory().percent
    except Exception:
        return None


def get_cpu_freq():
    try:
        freq = psutil.cpu_freq()
        return round(freq.current, 2) if freq else None
    except Exception:
        return None


def get_memory_footprint_mb():
    try:
        return round(psutil.Process(os.getpid()).memory_info().rss / (1024 * 1024), 4)
    except Exception:
        return None


def get_cpu_temp():
    try:
        temps = psutil.sensors_temperatures()
        if not temps:
            return None
        for key in ("coretemp", "k10temp", "cpu_thermal", "acpitz"):
            if key in temps:
                vals = [x.current for x in temps[key] if x.current and x.current > 0]
                if vals:
                    return round(sum(vals) / len(vals), 1)
    except Exception:
        pass
    return None


def get_cpu_power_draw_w():
    return None


def get_cpu_cores_used():
    try:
        return sum(1 for p in psutil.cpu_percent(percpu=True) if p > 1.0)
    except Exception:
        return None

# ============================================================
# PATH / ARTIFACT DISCOVERY
# ============================================================

def first_existing(paths: list[Path], description: str) -> Path:
    for path in paths:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find {description}. Tried:\n" + "\n".join(str(p) for p in paths)
    )


def score_artifact(path: Path, required_words: tuple[str, ...], preferred_words: tuple[str, ...]) -> int:
    name = path.name.lower()
    if any(word not in name for word in required_words):
        return -1
    return sum(10 for word in preferred_words if word in name) - len(name) // 100


def find_best_file(root: Path, suffix: str, required_words=(), preferred_words=()) -> Path:
    candidates = [p for p in root.rglob(f"*{suffix}") if p.is_file()]
    ranked = []
    for p in candidates:
        s = score_artifact(p, tuple(required_words), tuple(preferred_words))
        if s >= 0:
            ranked.append((s, p))
    if not ranked:
        raise FileNotFoundError(
            f"No matching {suffix} artifact found under {root}. "
            f"Required words={required_words}"
        )
    ranked.sort(key=lambda x: (x[0], x[1].stat().st_mtime), reverse=True)
    return ranked[0][1]


def find_model_artifacts(result_root: Path):
    # Prefer q1-named artifacts, but tolerate earlier naming variations.
    try:
        svm_path = find_best_file(
            result_root, ".joblib",
            required_words=("svm",),
            preferred_words=("q1",),
        )
    except FileNotFoundError:
        svm_path = find_best_file(result_root, ".joblib", preferred_words=("q1",))

    try:
        train_states_path = find_best_file(
            result_root, ".npy",
            required_words=("train", "states"),
            preferred_words=("q1",),
        )
    except FileNotFoundError:
        train_states_path = find_best_file(
            result_root, ".npy",
            required_words=("states",),
            preferred_words=("train", "q1"),
        )

    # NPZ metadata is optional for testing, but useful for validation.
    npz_files = list(result_root.rglob("*.npz"))
    model_npz_path = None
    if npz_files:
        model_npz_path = sorted(
            npz_files,
            key=lambda p: (("q1" in p.name.lower()), p.stat().st_mtime),
            reverse=True,
        )[0]

    return svm_path, train_states_path, model_npz_path

# ============================================================
# QASM DISCOVERY / LABEL MATCHING
# ============================================================

def is_qasm_file(path: Path) -> bool:
    if not path.is_file():
        return False
    if path.suffix.lower() == ".qasm":
        return True
    try:
        beginning = path.read_text(encoding="utf-8", errors="ignore")[:300].lower()
    except OSError:
        return False
    return "openqasm" in beginning and "qreg" in beginning


def numeric_identifier(path: Path) -> str | None:
    matches = re.findall(r"\d+", path.stem)
    return matches[-1] if matches else None


def read_label(path: Path) -> int:
    text = path.read_text(encoding="utf-8", errors="ignore").strip()
    matches = re.findall(r"-?\d+", text)
    if not matches:
        raise ValueError(f"No integer label found in {path}")
    label = int(matches[0])
    if label not in range(N_CLASSES):
        raise ValueError(f"Invalid label {label} in {path}")
    return label


def discover_test_samples(data_root: Path, config: dict) -> pd.DataFrame:
    # Usually data lives in an extracted/ subfolder, but search recursively from data_root.
    all_files = [p for p in data_root.rglob("*") if p.is_file()]
    qasm_all = [p for p in all_files if is_qasm_file(p)]

    archive_tokens = [x.lower() for x in config["archive_tokens"]]
    dataset_tokens = [x.lower() for x in config["dataset_tokens"]]

    def path_matches_test(p: Path) -> bool:
        s = str(p).lower()
        if any(tok in s for tok in archive_tokens):
            return True
        return "test" in s and any(tok in s for tok in dataset_tokens) and FIDELITY.lower() in s

    qasm_files = [p for p in qasm_all if path_matches_test(p)]
    if not qasm_files:
        # Last-resort fallback: all QASM files containing test.
        qasm_files = [p for p in qasm_all if "test" in str(p).lower()]
    if not qasm_files:
        raise FileNotFoundError(f"No test QASM files found under {data_root}")

    label_files = []
    for p in all_files:
        s = str(p).lower()
        if "label" not in s:
            continue
        if any(tok in s for tok in archive_tokens) or (
            "test" in s and any(tok in s for tok in dataset_tokens)
        ):
            label_files.append(p)

    if not label_files:
        raise FileNotFoundError(f"No test label files found under {data_root}")

    label_index = {}
    for p in sorted(label_files):
        ident = numeric_identifier(p)
        if ident is not None:
            label_index.setdefault(ident, p)

    records = []
    unmatched = 0
    for qasm_path in qasm_files:
        ident = numeric_identifier(qasm_path)
        if ident is None or ident not in label_index:
            unmatched += 1
            continue
        try:
            label = read_label(label_index[ident])
        except Exception:
            unmatched += 1
            continue
        records.append({
            "sample_id": ident,
            "qasm_path": str(qasm_path),
            "label_path": str(label_index[ident]),
            "label": label,
        })

    df = pd.DataFrame(records)
    if df.empty:
        raise RuntimeError(
            f"QASM files were found under {data_root}, but no QASM-label pairs were created."
        )

    if unmatched:
        print(f"Warning: {unmatched} test QASM files were unmatched in {data_root.name}.")

    return df


def select_ten_balanced(df: pd.DataFrame, seed: int) -> pd.DataFrame:
    # One sample per class when SAMPLES_PER_DATASET == 10.
    if SAMPLES_PER_DATASET == N_CLASSES:
        groups = []
        for cls in range(N_CLASSES):
            rows = df[df["label"] == cls]
            if rows.empty:
                raise ValueError(f"Class {cls} has no available test samples.")
            groups.append(rows.sample(n=1, random_state=seed + cls))
        return pd.concat(groups, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)

    # General deterministic balanced-ish fallback.
    return df.sample(n=min(SAMPLES_PER_DATASET, len(df)), random_state=seed).reset_index(drop=True)

# ============================================================
# QASM PARSER / EXECUTOR
# ============================================================

def remove_qasm_comments(qasm_text: str) -> str:
    qasm_text = re.sub(r"/\*.*?\*/", "", qasm_text, flags=re.DOTALL)
    return re.sub(r"//.*?$", "", qasm_text, flags=re.MULTILINE)


def split_qasm_statements(qasm_text: str) -> list[str]:
    return [s.strip() for s in remove_qasm_comments(qasm_text).split(";") if s.strip()]


def safe_qasm_angle(expression: str) -> float:
    expression = expression.strip().replace("^", "**")
    if not re.fullmatch(r"[0-9eEpiPI+\-*/().\s*]+", expression):
        raise ValueError(f"Unsupported QASM parameter expression: {expression}")
    value = eval(expression, {"__builtins__": {}}, {"pi": np.pi, "PI": np.pi})
    value = float(value)
    if not np.isfinite(value):
        raise ValueError(f"Non-finite QASM parameter: {expression}")
    return value


def parse_parameter_list(text: str | None) -> list[float]:
    if text is None or not text.strip():
        return []
    return [safe_qasm_angle(x) for x in text.split(",")]


def parse_wire_list(operand_text: str, register_sizes: dict[str, int]) -> list[int]:
    offsets = {}
    running = 0
    for name, size in register_sizes.items():
        offsets[name] = running
        running += size

    wires = []
    for operand in [x.strip() for x in operand_text.split(",") if x.strip()]:
        match = re.fullmatch(r"([A-Za-z_]\w*)\s*\[\s*(\d+)\s*\]", operand)
        if match is None:
            raise ValueError(f"Unsupported QASM qubit operand: {operand}")
        reg, idx = match.group(1), int(match.group(2))
        if reg not in register_sizes or not 0 <= idx < register_sizes[reg]:
            raise ValueError(f"Invalid QASM qubit operand: {operand}")
        wires.append(offsets[reg] + idx)
    return wires


def parse_mnisq_qasm(qasm_text: str):
    statements = split_qasm_statements(qasm_text)
    register_sizes = {}
    raw_gates = []
    ignored_prefixes = ("openqasm", "include", "creg", "measure", "barrier", "reset")

    for statement in statements:
        lowered = statement.lower().strip()
        qreg_match = re.fullmatch(
            r"qreg\s+([A-Za-z_]\w*)\s*\[\s*(\d+)\s*\]",
            statement,
            flags=re.IGNORECASE,
        )
        if qreg_match:
            register_sizes[qreg_match.group(1)] = int(qreg_match.group(2))
            continue
        if lowered.startswith(ignored_prefixes):
            continue
        if lowered.startswith(("gate ", "opaque ")):
            raise ValueError("Custom gate declarations were found.")
        raw_gates.append(statement)

    if not register_sizes:
        raise ValueError("No qreg declaration found.")

    total_qubits = sum(register_sizes.values())
    if total_qubits != SOURCE_QUBITS:
        raise ValueError(
            f"Source circuit declares {total_qubits} qubits; expected {SOURCE_QUBITS}."
        )

    gate_pattern = re.compile(
        r"^([A-Za-z_]\w*)(?:\s*\((.*?)\))?\s+(.+)$",
        flags=re.DOTALL,
    )
    operations = []
    for statement in raw_gates:
        match = gate_pattern.fullmatch(statement.strip())
        if match is None:
            raise ValueError(f"Could not parse QASM statement: {statement}")
        operations.append((
            match.group(1).lower(),
            parse_parameter_list(match.group(2)),
            parse_wire_list(match.group(3), register_sizes),
        ))
    return operations


def require_shape(name, params, wires, n_params, n_wires):
    if len(params) != n_params or len(wires) != n_wires:
        raise ValueError(
            f"Gate {name} expects {n_params} parameter(s) and {n_wires} wire(s); "
            f"got {len(params)} and {len(wires)}."
        )


def apply_qasm_gate(name, p, w):
    if name in {"u", "u3"}:
        require_shape(name, p, w, 3, 1); qml.U3(p[0], p[1], p[2], wires=w[0])
    elif name == "u2":
        require_shape(name, p, w, 2, 1); qml.U3(np.pi / 2, p[0], p[1], wires=w[0])
    elif name in {"u1", "p", "phase"}:
        require_shape(name, p, w, 1, 1); qml.PhaseShift(p[0], wires=w[0])
    elif name == "rx":
        require_shape(name, p, w, 1, 1); qml.RX(p[0], wires=w[0])
    elif name == "ry":
        require_shape(name, p, w, 1, 1); qml.RY(p[0], wires=w[0])
    elif name == "rz":
        require_shape(name, p, w, 1, 1); qml.RZ(p[0], wires=w[0])
    elif name == "x":
        require_shape(name, p, w, 0, 1); qml.PauliX(wires=w[0])
    elif name == "y":
        require_shape(name, p, w, 0, 1); qml.PauliY(wires=w[0])
    elif name == "z":
        require_shape(name, p, w, 0, 1); qml.PauliZ(wires=w[0])
    elif name == "h":
        require_shape(name, p, w, 0, 1); qml.Hadamard(wires=w[0])
    elif name == "s":
        require_shape(name, p, w, 0, 1); qml.S(wires=w[0])
    elif name == "sdg":
        require_shape(name, p, w, 0, 1); qml.adjoint(qml.S)(wires=w[0])
    elif name == "t":
        require_shape(name, p, w, 0, 1); qml.T(wires=w[0])
    elif name == "tdg":
        require_shape(name, p, w, 0, 1); qml.adjoint(qml.T)(wires=w[0])
    elif name == "sx":
        require_shape(name, p, w, 0, 1); qml.SX(wires=w[0])
    elif name == "sxdg":
        require_shape(name, p, w, 0, 1); qml.adjoint(qml.SX)(wires=w[0])
    elif name in {"id", "i"}:
        require_shape(name, p, w, 0, 1); qml.Identity(wires=w[0])
    elif name in {"cx", "cnot"}:
        require_shape(name, p, w, 0, 2); qml.CNOT(wires=w)
    elif name == "cy":
        require_shape(name, p, w, 0, 2); qml.CY(wires=w)
    elif name == "cz":
        require_shape(name, p, w, 0, 2); qml.CZ(wires=w)
    elif name == "swap":
        require_shape(name, p, w, 0, 2); qml.SWAP(wires=w)
    elif name == "crx":
        require_shape(name, p, w, 1, 2); qml.CRX(p[0], wires=w)
    elif name == "cry":
        require_shape(name, p, w, 1, 2); qml.CRY(p[0], wires=w)
    elif name == "crz":
        require_shape(name, p, w, 1, 2); qml.CRZ(p[0], wires=w)
    elif name in {"cp", "cu1"}:
        require_shape(name, p, w, 1, 2); qml.ControlledPhaseShift(p[0], wires=w)
    elif name in {"ccx", "toffoli"}:
        require_shape(name, p, w, 0, 3); qml.Toffoli(wires=w)
    else:
        raise ValueError(f"Unsupported QASM gate: {name}")


def reduce_to_q1(source_state: np.ndarray, target_wire: int = 0) -> np.ndarray:
    source_state = np.asarray(source_state, dtype=np.complex128).reshape([2] * SOURCE_QUBITS)
    psi = np.moveaxis(source_state, target_wire, 0).reshape(2, -1)
    rho = psi @ psi.conj().T
    rho = (rho + rho.conj().T) / 2.0
    vals, vecs = np.linalg.eigh(rho)
    q1 = vecs[:, np.argmax(vals)]
    nz = np.flatnonzero(np.abs(q1) > 1e-12)
    if len(nz):
        q1 = q1 * np.exp(-1j * np.angle(q1[nz[0]]))
    norm = np.linalg.norm(q1)
    if not np.isfinite(norm) or norm <= 0:
        raise RuntimeError("Invalid q=1 reduced state.")
    return q1 / norm


def execute_qasm_to_q1(qasm_path: Path, cache_root: Path) -> np.ndarray:
    cache_root.mkdir(parents=True, exist_ok=True)
    stat = qasm_path.stat()
    key = f"{qasm_path.resolve()}|{stat.st_size}|{stat.st_mtime_ns}|{PENNYLANE_DEVICE}|q1"
    cache_path = cache_root / f"{hashlib.sha256(key.encode()).hexdigest()}.npy"
    if cache_path.exists():
        x = np.load(cache_path, allow_pickle=False)
        if x.shape == (2,):
            return x

    qasm_text = qasm_path.read_text(encoding="utf-8", errors="ignore")
    operations = parse_mnisq_qasm(qasm_text)
    dev = qml.device(PENNYLANE_DEVICE, wires=SOURCE_QUBITS, shots=None)

    @qml.qnode(dev, interface=None, diff_method=None)
    def circuit_state():
        for name, params, wires in operations:
            apply_qasm_gate(name, params, wires)
        return qml.state()

    source_state = np.asarray(circuit_state(), dtype=np.complex128).reshape(-1)
    if source_state.size != 2 ** SOURCE_QUBITS:
        raise RuntimeError(f"Unexpected source state dimension: {source_state.size}")
    source_state /= np.linalg.norm(source_state)
    q1 = reduce_to_q1(source_state)
    np.save(cache_path, q1, allow_pickle=False)
    return q1

# ============================================================
# KERNEL / PREDICTION QUALITY / ENERGY
# ============================================================

def fidelity_kernel_row(q1_state: np.ndarray, train_states: np.ndarray) -> np.ndarray:
    overlaps = q1_state.conj().reshape(1, -1) @ train_states.T
    return np.clip(np.abs(overlaps) ** 2, 0.0, 1.0)


def prediction_quality(classifier, kernel_row):
    confidence = None
    margin = None
    entropy = None
    try:
        d = np.asarray(classifier.decision_function(kernel_row)).reshape(-1)
        if d.size >= 2:
            top2 = np.sort(d)[-2:]
            margin = float(top2[-1] - top2[-2])
        elif d.size == 1:
            margin = float(abs(d[0]))
    except Exception:
        pass
    try:
        p = np.asarray(classifier.predict_proba(kernel_row)).reshape(-1)
        if p.size:
            p = np.clip(p.astype(float), 1e-12, 1.0)
            confidence = float(np.max(p))
            entropy = float(-np.sum(p * np.log(p)))
    except Exception:
        pass
    return (
        round(confidence, 6) if confidence is not None else None,
        round(margin, 6) if margin is not None else None,
        round(entropy, 6) if entropy is not None else None,
    )


def extract_energy_data(tracker, emissions_value):
    fd = getattr(tracker, "final_emissions_data", None)
    cpu = getattr(fd, "cpu_energy", 0) if fd else 0
    gpu = getattr(fd, "gpu_energy", 0) if fd else 0
    ram = getattr(fd, "ram_energy", 0) if fd else 0
    total = getattr(fd, "energy_consumed", 0) if fd else 0
    ci = None
    if emissions_value is not None and total and total > 0:
        ci = float(emissions_value) / float(total)
    return cpu or 0, gpu or 0, ram or 0, total or 0, ci


def predict_with_energy(classifier, kernel_row, dataset_name):
    tracker = None
    if CODECARBON_AVAILABLE:
        try:
            tracker = EmissionsTracker(
                project_name=f"mnisq_{dataset_name}_q1_fingerprinting",
                output_dir=str(OUTPUT_ROOT / "codecarbon"),
                output_file=f"{dataset_name.lower().replace('-', '_')}.csv",
                log_level="error",
                save_to_file=True,
            )
            tracker.start()
        except Exception:
            tracker = None

    t0 = time.perf_counter()
    pred = int(classifier.predict(kernel_row)[0])
    exec_time = time.perf_counter() - t0

    if tracker is not None:
        try:
            emissions = tracker.stop()
            cpu_e, gpu_e, ram_e, total_e, ci = extract_energy_data(tracker, emissions)
        except Exception:
            emissions, cpu_e, gpu_e, ram_e, total_e, ci = 0, 0, 0, 0, 0, None
    else:
        emissions, cpu_e, gpu_e, ram_e, total_e, ci = 0, 0, 0, 0, 0, None

    return pred, exec_time, cpu_e, gpu_e, ram_e, total_e, emissions, ci

# ============================================================
# TELEMETRY ROW
# ============================================================

def build_row(
    dataset_name,
    model_name,
    sample_index,
    sample_id,
    qasm_path,
    true_label,
    prediction,
    exec_time,
    confidence,
    margin,
    entropy,
    cpu_energy,
    gpu_energy,
    ram_energy,
    total_energy,
    emissions,
    carbon_intensity,
    svm_path,
    train_states_path,
):
    input_tokens = 784
    output_tokens = 1
    total_tokens = input_tokens + output_tokens

    energy_per_token_kwh = 0.0
    joules_per_token = 0.0
    watts_estimated = 0.0
    gpu_energy_pct = 0.0
    cpu_energy_pct = 0.0

    if total_energy and total_energy > 0:
        energy_per_token_kwh = total_energy / total_tokens
        joules_per_token = (total_energy * 3_600_000) / total_tokens
        if exec_time > 0:
            watts_estimated = (total_energy * 3_600_000) / exec_time
        gpu_energy_pct = (gpu_energy / total_energy) * 100
        cpu_energy_pct = (cpu_energy / total_energy) * 100

    return {
        # Identity
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "unique_device_id": DEVICE_UUID,
        "device_short_id": DEVICE_SHORT,
        "pc_name": get_hostname(),
        "collection_mode": "automated_edge",

        # Sample
        "dataset": dataset_name,
        "sample_index": sample_index,
        "sample_id": sample_id,
        "qasm_path": qasm_path,
        "true_label": int(true_label),
        "prediction": int(prediction),
        "correct": int(prediction) == int(true_label),

        # Model identity
        "model_type": model_name,
        "parameters": None,
        "model_flops": None,
        "svm_model_path": str(svm_path),
        "train_states_path": str(train_states_path),

        # Prediction quality
        "confidence_score": confidence,
        "logit_margin": margin,
        "entropy": entropy,

        # Timing
        "execution_time_sec": round(exec_time, 10),

        # Energy
        "cpu_energy_kwh": cpu_energy,
        "gpu_energy_kwh": gpu_energy,
        "ram_energy_kwh": ram_energy,
        "total_energy_kwh": total_energy,
        "total_emissions_kg": emissions,
        "carbon_intensity_kgco2_kwh": carbon_intensity,
        "codecarbon_version": CODECARBON_VERSION,

        # Efficiency
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": total_tokens,
        "tokens_per_second": round(total_tokens / exec_time, 4) if exec_time > 0 else None,
        "joules_per_token": round(joules_per_token, 6),
        "energy_per_token_kwh": round(energy_per_token_kwh, 12),
        "watts_estimated": round(watts_estimated, 4),
        "gpu_energy_pct_of_total": round(gpu_energy_pct, 2),
        "cpu_energy_pct_of_total": round(cpu_energy_pct, 2),

        # CPU
        "cpu_model": CPU_MODEL_NAME,
        "cpu_architecture": CPU_ARCH,
        "cpu_core_count": CPU_CORE_COUNT,
        "cpu_thread_count": CPU_THREAD_COUNT,
        "cpu_core": CPU_CORE_COUNT,
        "cpu_thread": CPU_THREAD_COUNT,
        "cpu_tdp_w": CPU_TDP_W,
        "cpu_usage_pct": get_cpu_usage(),
        "cpu_clock_mhz": get_cpu_freq(),
        "cpu_temp_c": get_cpu_temp(),
        "cpu_power_draw_w": get_cpu_power_draw_w(),
        "cpu_cores_used": get_cpu_cores_used(),

        # GPU: default.qubit CPU simulation
        "gpu_model": "No GPU used by default.qubit",
        "gpu_core": None,
        "gpu_thread": None,
        "gpu_driver_version": None,
        "gpu_compute_capability": None,
        "gpu_power_limit_w": None,
        "gpu_memory_total_mb": None,
        "gpu_power_draw_w": None,
        "gpu_utilization_pct": None,
        "gpu_temp_c": None,
        "gpu_memory_used_mb": None,
        "gpu_sm_clock_mhz": None,
        "gpu_memory_clock_mhz": None,
        "cuda_driver_version": None,
        "cuda_available": False,
        "device_type": PENNYLANE_DEVICE,

        # RAM / environment
        "ram_usage_pct": get_ram_usage(),
        "memory_footprint_mb": get_memory_footprint_mb(),
        "system_ram_total_gb": SYSTEM_RAM_TOTAL_GB,
        "os_name": OS_NAME,
        "os_version": OS_VERSION,
        "os_architecture": OS_ARCHITECTURE,
        "os_full_name": OS_FULL_NAME,
        "python_version": PYTHON_VERSION,
        "torch_version": None,

        # Final model metrics: backfilled per dataset
        "model_accuracy": None,
        "model_precision_weighted": None,
        "model_recall_weighted": None,
        "model_f1_weighted": None,

        # Custom / quantum context
        "quantum_computing": 1,
        "model_under_attack": 0,
        "source_qubits": SOURCE_QUBITS,
        "n_qubits": N_QUBITS,
        "fidelity": FIDELITY,
        "pennylane_device": PENNYLANE_DEVICE,
        "pennylane_version": PENNYLANE_VERSION,
    }

# ============================================================
# DATASET TEST
# ============================================================

def test_one_dataset(dataset_name: str, config: dict) -> pd.DataFrame:
    print("\n" + "=" * 78)
    print(f"Testing {dataset_name}: {SAMPLES_PER_DATASET} q=1 fingerprinting samples")
    print("=" * 78)

    data_root = first_existing(config["data_roots"], f"{dataset_name} data folder")
    result_root = first_existing(config["result_roots"], f"{dataset_name} q1 results folder")

    svm_path, train_states_path, model_npz_path = find_model_artifacts(result_root)
    print(f"Data root       : {data_root}")
    print(f"Results root    : {result_root}")
    print(f"SVM             : {svm_path.name}")
    print(f"Training states : {train_states_path.name}")
    if model_npz_path:
        print(f"Model metadata  : {model_npz_path.name}")

    classifier = joblib.load(svm_path)
    train_states = np.load(train_states_path, allow_pickle=False)

    if train_states.ndim != 2 or train_states.shape[1] != 2 ** N_QUBITS:
        raise ValueError(
            f"{dataset_name}: expected q=1 training states with shape (N, 2), "
            f"but found {train_states.shape} in {train_states_path}."
        )

    discovered = discover_test_samples(data_root, config)
    selected = select_ten_balanced(discovered, seed=SEED)
    print("Selected class labels:", selected["label"].tolist())

    selected.to_csv(
        OUTPUT_ROOT / f"selected_{dataset_name.lower().replace('-', '_')}_10.csv",
        index=False,
    )

    cache_root = result_root / "fingerprinting_test_state_cache_q1"
    rows = []

    for i, sample in tqdm(
        selected.iterrows(),
        total=len(selected),
        desc=f"{dataset_name} fingerprinting",
        unit="sample",
    ):
        qasm_path = Path(sample["qasm_path"])

        # Circuit simulation + q1 reduction is required to build the kernel row.
        # execution_time_sec below intentionally measures classifier prediction,
        # matching the user's earlier telemetry pattern.
        q1_state = execute_qasm_to_q1(qasm_path, cache_root)
        kernel_row = fidelity_kernel_row(q1_state, train_states)

        confidence, margin, entropy = prediction_quality(classifier, kernel_row)

        pred, exec_time, cpu_e, gpu_e, ram_e, total_e, emissions, ci = predict_with_energy(
            classifier,
            kernel_row,
            dataset_name,
        )

        rows.append(build_row(
            dataset_name=dataset_name,
            model_name=f"MNISQ_{dataset_name}_QuantumKernelSVM_q1",
            sample_index=i,
            sample_id=sample["sample_id"],
            qasm_path=str(qasm_path),
            true_label=int(sample["label"]),
            prediction=pred,
            exec_time=exec_time,
            confidence=confidence,
            margin=margin,
            entropy=entropy,
            cpu_energy=cpu_e,
            gpu_energy=gpu_e,
            ram_energy=ram_e,
            total_energy=total_e,
            emissions=emissions,
            carbon_intensity=ci,
            svm_path=svm_path,
            train_states_path=train_states_path,
        ))

    df = pd.DataFrame(rows)

    # Backfill model-level metrics into all 10 rows for this dataset.
    y_true = df["true_label"].astype(int)
    y_pred = df["prediction"].astype(int)
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)

    df["model_accuracy"] = float(acc)
    df["model_precision_weighted"] = float(prec)
    df["model_recall_weighted"] = float(rec)
    df["model_f1_weighted"] = float(f1)

    dataset_csv = DEVICE_LOG_DIR / f"mnisq_q1_fingerprint_{dataset_name.lower().replace('-', '_')}_10.csv"
    df.to_csv(dataset_csv, index=False)

    print(
        f"{dataset_name} -> Accuracy={acc:.4f}, Precision={prec:.4f}, "
        f"Recall={rec:.4f}, F1={f1:.4f}"
    )
    print(f"Saved: {dataset_csv}")

    return df

# ============================================================
# MAIN
# ============================================================

def main():
    print("=" * 78)
    print("MNISQ q=1 MULTI-DATASET FINGERPRINTING TEST")
    print("Datasets: MNIST, FashionMNIST, Kuzushiji-MNIST")
    print(f"Samples per dataset: {SAMPLES_PER_DATASET}")
    print(f"Expected total rows: {SAMPLES_PER_DATASET * len(DATASET_CONFIGS)}")
    print(f"Device ID: {DEVICE_SHORT}")
    print("=" * 78)

    all_results = []
    failures = []

    for dataset_name, config in DATASET_CONFIGS.items():
        try:
            all_results.append(test_one_dataset(dataset_name, config))
        except Exception as exc:
            failures.append({"dataset": dataset_name, "error": repr(exc)})
            print(f"\n[{dataset_name} FAILED] {type(exc).__name__}: {exc}")

    if all_results:
        combined = pd.concat(all_results, ignore_index=True)
        combined_path = DEVICE_LOG_DIR / "mnisq_q1_fingerprinting_all_3datasets_30samples.csv"
        combined.to_csv(combined_path, index=False)
        print("\n" + "=" * 78)
        print(f"Combined rows saved: {len(combined)}")
        print(f"Combined CSV: {combined_path}")

        summary = (
            combined.groupby("dataset")
            .agg(
                samples=("prediction", "size"),
                accuracy=("correct", "mean"),
                avg_execution_time_sec=("execution_time_sec", "mean"),
                avg_total_energy_kwh=("total_energy_kwh", "mean"),
            )
            .reset_index()
        )
        summary_path = DEVICE_LOG_DIR / "mnisq_q1_fingerprinting_summary.csv"
        summary.to_csv(summary_path, index=False)
        print(f"Summary CSV : {summary_path}")
        print("\nSummary:")
        print(summary.to_string(index=False))

    if failures:
        failure_path = DEVICE_LOG_DIR / "mnisq_q1_fingerprinting_failures.csv"
        pd.DataFrame(failures).to_csv(failure_path, index=False)
        print(f"\nFailures saved: {failure_path}")

    if not all_results:
        raise RuntimeError("All three dataset tests failed. Check the failure CSV above.")

    print("\nDone.")


if __name__ == "__main__":
    main()
